# try-except-solve — worked example 1: Safe solve returning None on singular

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `try-except-solve`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

`torch.linalg.solve(A, b)` raises a `RuntimeError` (a `LinAlgError` subclass) when `A` is singular. The unbatched graceful pattern wraps the call in `try/except RuntimeError` and returns `None` so the caller can branch on a clean Python sentinel instead of crashing.

## Worked solution

We want a solve that returns the solution for invertible `A` and `None` for a singular `A`.

1. Inside `try`, we call `t.linalg.solve(A, b)`. For a well-conditioned `A` this returns the solution vector and we return it directly.
2. We catch `RuntimeError`, which is the base class PyTorch raises (its `_LinAlgError` is a subclass), so a singular matrix lands in the `except` branch.
3. In that branch we return `None`, signalling 'no solution' without propagating the exception.

We exercise both paths: an invertible system gives a real vector, and a rank-deficient matrix returns `None`.

In [ ]:
import torch as t
from typing import Optional

def safe_solve(A: t.Tensor, b: t.Tensor) -> Optional[t.Tensor]:
    try:
        return t.linalg.solve(A, b)
    except RuntimeError:
        return None

A_good = t.tensor([[2.0, 0.0], [0.0, 4.0]])
b = t.tensor([6.0, 8.0])
A_sing = t.tensor([[1.0, 2.0], [2.0, 4.0]])  # rows linearly dependent
print('good:', safe_solve(A_good, b).tolist())
print('singular:', safe_solve(A_sing, b))